In [1]:
import numpy as np 
import pandas as pd 
import os  
import sys
import plotly.graph_objects as go 

In [2]:
OR_LEARNING_PATH = os.path.join(os.getcwd().split('OR_learning')[0], 'OR_learning/')
sys.path.insert(0, os.path.join(OR_LEARNING_PATH, 'utils/'))

import SequenceAlignment_functions as sa
import plot_functions as pf
import color_function as cf 
import pdb_functions as pu

### Query pS6-IP data

The goal is to identify OR-ligand pairs for generate AF3 structure with. <br>
Ultimately, want to create a database of AF3 structures with positively interacting as well as none interacting OR-ligand pairs 

In [2]:
ps6_df = pd.read_csv('/mnt/data2/Justice/OR_learning/files/pS6IP/pS6IP_MASTER_HL_Annotated_2025.csv')
ps6_df = ps6_df[['DL_OR', 'odor', 'concentration', 'odor_and_conc', 'logFC', 'FDR']]

In [ ]:
ps6_df[ps6_df['DL_OR'] == 'Or1E1C'].sort_values(['FDR', 'logFC'], ascending=[True, False])

In [38]:
plot_data = ps6_df[ps6_df['DL_OR'] == 'Or1E1C']

fig = go.Figure()
fig.add_traces(go.Scatter(
    x = plot_data['logFC'],
    y = plot_data['FDR'].apply(lambda x: -np.log(x)), 
    text = plot_data['DL_OR'], 
    mode = 'markers'
    )
)
fig.update_layout(template = 'simple_white')
fig.show()

In [238]:
ps6_df

,DL_OR,cid,odor,concentration,odor_and_conc,odor_category,logFC,FDR
0,Or11H4,6054,2-Phenylethanol,1p,1% 2-Phenylethanol,Alcohols,2.544864e+00,6.676352e-65
1,Or11G2,6054,2-Phenylethanol,1p,1% 2-Phenylethanol,Alcohols,2.472033e+00,2.460729e-21
2,Or8U9,6054,2-Phenylethanol,1p,1% 2-Phenylethanol,Alcohols,2.164904e+00,2.596337e-17
3,Or6N1,6054,2-Phenylethanol,1p,1% 2-Phenylethanol,Alcohols,1.770826e+00,2.680831e-09
4,Or11H6,6054,2-Phenylethanol,1p,1% 2-Phenylethanol,Alcohols,1.014952e+00,1.436028e-06
...,...,...,...,...,...,...,...,...
80441,Or5P54,68490,2-Hydroxy acetophenone,1p,1% 2-Hydroxy acetophenone,Ketones,6.844914e-17,1.000000e+00
80442,Or8B36,68490,2-Hydroxy acetophenone,1p,1% 2-Hydroxy acetophenone,Ketones,6.844914e-17,1.000000e+00
80443,Or14J8,68490,2-Hydroxy acetophenone,1p,1% 2-Hydroxy acetophenone,Ketones,6.844914e-17,1.000000e+00
80444,Or56B2J,68490,2-Hydroxy acetophenone,1p,1% 2-Hydroxy acetophenone,Ketones,6.844914e-17,1.000000e+00


In [242]:
plot_data = ps6_df[ps6_df['odor_and_conc'] == '1% Pyridine']

fig = go.Figure()

fig.add_traces(go.Scatter(
    x = plot_data['logFC'],
    y = plot_data['FDR'].apply(lambda x: -np.log(x)), 
    text = plot_data['DL_OR'], 
    mode = 'markers'
    )
)
fig.update_layout(template = 'simple_white')
fig.show()

In [ ]:
import pandas as pd

# Load your dataframe
df = ps6_df.copy()

# Step 1: Filter Positive Pairs
positive_df = df[(df["FDR"] <= 0.05)]  # Keep FDR ≤ 0.05
positive_df = positive_df.sort_values(by="logFC", ascending=False)  # Maximize logFC

# Remove ligands that activate too many ORs
ligand_counts = positive_df["odor"].value_counts()
threshold = ligand_counts.quantile(0.9)  # Top 10% most activating ligands removed
excluded_ligands = ligand_counts[ligand_counts > threshold].index
positive_df = positive_df[~positive_df["odor"].isin(excluded_ligands)]

# Step 2: Identify Negative Pairs
negative_df = df.copy()

# Remove positive OR-ligand pairs from negative set
negative_df = negative_df[~negative_df.set_index(["DL_OR", "odor"]).index.isin(
    positive_df.set_index(["DL_OR", "odor"]).index
)]

# Keep only high FDR values (ensuring the OR does not respond to these ligands)
negative_df = negative_df[negative_df["FDR"] > 0.5]  # Adjust threshold if needed

# Save the results
positive_df.to_csv("positive_pairs.csv", index=False)
negative_df.to_csv("negative_pairs.csv", index=False)

print("Filtered positive and negative OR-ligand pairs saved.")

# OR-Ligand Pair Selection for AlphaFold3 generation

 Selects **positive** and **negative** OR-ligand pairs based on FDR and logFC thresholds to create a reliable dataset for model training.  

### **Positive OR-Ligand Selection**  
We define **positive responses** as OR-ligand pairs where:  
- FDR ≤ 0.05 (high confidence response)  
- logFC > 0 (ensures activation, not repression)  
- Concentration is NOT `'100p'` or `'10p'` (excluding potentially unreliable high concentrations)  

The **top 4 odors** are chosen based on the highest number of positive responses.  
For each odor, the **top 5 ORs** are selected based on low FDR and high logFC

### **Negative OR-Ligand Selection**
Negative pairs are selected from ORs that were previously positive responders to any ligand.
For these ORs, a negative response is identified as:
- FDR > 0.05 (ensuring confident non-response)
- Concentration is NOT `'100p'` or `'10p'`

For simplicity, the negative OR's odor are simply selected using the same odor as pos OR


In [41]:
ps6_df = pd.read_csv('/mnt/data2/Justice/OR_learning/files/pS6IP/pS6IP_MASTER_HL_Annotated_2025.csv')
ps6_df = ps6_df[['DL_OR', 'cid', 'odor', 'concentration', 'odor_and_conc', 'logFC', 'FDR']]

df = ps6_df.copy()

positive_df = df[(df["FDR"] <= 0.05) & 
                 (df['logFC'] > 0)   & 
                #  (~df['concentration'].isin(['100p', '10p']))
                 (df['concentration'].isin(['1p']))
                 ]  
positive_df = positive_df.sort_values(by="logFC", ascending=False) 
# positive_df["odor_and_conc"].value_counts()


# Selecting for pos ORs. 
top_odors = list(positive_df["odor_and_conc"].value_counts().head(5).index)
# Include manully picked odors 
top_odors += ['1% (+)-Menthol', '1% (-)-Menthol']

filtered_df = positive_df[positive_df["odor_and_conc"].isin(top_odors)]
filtered_df = filtered_df.sort_values(by=["odor_and_conc", "FDR", "logFC"], ascending=[True, True, False])
top_pos_ORs = filtered_df.groupby("odor_and_conc").head(5)


# Selecting for negative ORs. 
positive_ORs = top_pos_ORs["DL_OR"].unique()
negative_df = df[(df["DL_OR"].isin(positive_ORs)) & 
                 (df["odor_and_conc"].isin(top_pos_ORs['odor_and_conc'].unique())) & # Using same odor_and_conc for simplicity
                 (df["FDR"] > 0.5) &
                 (~df['concentration'].isin(['100p', '10p']))]
negative_df = negative_df.sort_values(by=["odor", "FDR"], ascending=[True, False])
# Filter by picking top 5 for each odor_and_conc
top_neg_ORs = negative_df.groupby("odor_and_conc", group_keys=False).apply(lambda x: x.nlargest(3, "FDR")) 


/tmp/ipykernel_224083/2076688939.py:33: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top_neg_ORs = negative_df.groupby("odor_and_conc", group_keys=False).apply(lambda x: x.nlargest(3, "FDR"))


In [ ]:
"""
Quickly read .fa and make sequnece.csv for easier parsing in the future 
"""
# from Bio.SeqIO.FastaIO import SimpleFastaParser

# def read_fasta(file_path):
#     """
#     Reads a FASTA file and returns a Pandas DataFrame.

#     Args:
#         file_path (str): The path to the FASTA file.

#     Returns:
#         pd.DataFrame: A DataFrame with columns 'id' and 'sequence'.
#     """
#     with open(file_path, 'r') as fasta_file:
#         records = []
#         for title, sequence in SimpleFastaParser(fasta_file):
#             records.append({'id': title.split()[0], 'sequence': sequence})
#     return pd.DataFrame(records)



# DL_OR = np.load('/mnt/data2/Justice/OR_learning/files/Olfr_DL.npy', allow_pickle=True).item()

# file_path = '/mnt/data2/Justice/OR_learning/files/mouseOR_alignment.fasta'
# fa = read_fasta(file_path)

# fa['DL_OR'] = fa['id'].apply(lambda x: DL_OR.get(x, x))
# fa['sequence'] = fa['sequence'].apply(lambda x : x.replace('-', ''))

# fa[['DL_OR', 'id', 'sequence']].to_csv('../../files/OR_seq/mOR_sequences.csv')

In [ ]:

# Merge sequence data with positive ORs
seq_df = pd.read_csv('/mnt/data2/Justice/OR_learning/files/OR_seq/mOR_sequences.csv', index_col = 0)
pos_merged = top_pos_ORs.merge(seq_df, on="DL_OR", how="left")
neg_merged = top_neg_ORs.merge(seq_df, on="DL_OR", how="left")
or_merged  = pd.concat([pos_merged, 
                        # neg_merged.sample(10, random_state=0)])
                        neg_merged])

# or_merged.to_csv('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/or_screen.csv')

In [43]:
"""
Make a representative plot for individual ORs picked from the pS6 data
"""
ps6_df = pd.read_csv('/mnt/data2/Justice/OR_learning/files/pS6IP/pS6IP_MASTER_HL_Annotated_2025.csv')
ps6_df = ps6_df[['DL_OR', 'odor', 'concentration', 'odor_and_conc', 'logFC', 'FDR']]

or_merged = pd.read_csv('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/or_screen.csv', 
                        index_col = 0)

In [ ]:
from plotly.subplots import make_subplots

num_cols = 3

# Get unique odor_and_conc values from or_merged
odor_conc_list = or_merged["odor_and_conc"].unique()
num_plots = len(odor_conc_list)

# Calculate the number of rows dynamically to form a square layout
num_rows = int(np.ceil(num_plots / num_cols))

# Create subplots with dynamic rows and fixed columns
fig = make_subplots(
    rows=num_rows, cols=num_cols,
    subplot_titles=[f"{odor}" for odor in odor_conc_list],
    shared_xaxes=False, shared_yaxes=False  # Ensures consistent axis scales
)

# Loop through each odor concentration and create scatter plots
for idx, _odor_conc in enumerate(odor_conc_list):
    row = (idx // num_cols) + 1  # Determine row index
    col = (idx % num_cols) + 1   # Determine column index

    # Filter data for the current odor concentration
    picked_OR = or_merged[or_merged["odor_and_conc"] == _odor_conc]
    plot_data = ps6_df[
        (ps6_df["odor_and_conc"] == _odor_conc) & 
        (~ps6_df["DL_OR"].isin(picked_OR["DL_OR"]))
    ]

    # Separate picked_OR into significant (FDR < 0.05) and non-significant (FDR >= 0.05)
    picked_significant = picked_OR[picked_OR["FDR"] < 0.05]
    picked_nonsignificant = picked_OR[picked_OR["FDR"] >= 0.05]

    # Scatter plot for other ORs (gray)
    fig.add_trace(
        go.Scatter(
            x=plot_data["logFC"],
            y=plot_data["FDR"].apply(lambda x: -np.log(x)), 
            text=plot_data["DL_OR"], 
            mode="markers",
            marker=dict(color="#D3D3D3", size=6, opacity=0.5),
            name="Other ORs",
            showlegend=(idx == 0)  # Show legend only in the first subplot
        ),
        row=row, col=col
    )
    
    # Scatter plot for significant picked ORs (red)
    fig.add_trace(
        go.Scatter(
            x=picked_significant["logFC"],
            y=picked_significant["FDR"].apply(lambda x: -np.log(x)), 
            text=picked_significant["DL_OR"], 
            mode="markers",
            marker=dict(color="red", size=8, opacity=0.9),
            name="Picked ORs (FDR < 0.05)",
            showlegend=(idx == 0)  # Show legend only in the first subplot
        ),
        row=row, col=col
    )

    # Scatter plot for non-significant picked ORs (blue)
    fig.add_trace(
        go.Scatter(
            x=picked_nonsignificant["logFC"],
            y=picked_nonsignificant["FDR"].apply(lambda x: -np.log(x)), 
            text=picked_nonsignificant["DL_OR"], 
            mode="markers",
            marker=dict(color="blue", size=8, opacity=0.9),
            name="Picked ORs (FDR ≥ 0.05)",
            showlegend=(idx == 0)  # Show legend only in the first subplot
        ),
        row=row, col=col
    )

# Update layout
fig.update_layout(
    height=400 * num_rows,  # Adjust height dynamically based on rows
    width=400 * num_cols,   # Adjust width dynamically based on columns
    title="pS6-IP OR picked for AF3 screening",
    template="simple_white",
    showlegend=True
)

fig.show()
fig.write_html('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/or_screen_vol.html')

#### Making AF3 .json files for individual ORs 

In [ ]:
"""
Take top_pos_ORs and top_neg_ORs write .json files for AF3 generation 
"""

import json
import pubchempy as pcp

def get_smiles(cid):
    try:
        compound = pcp.Compound.from_cid(cid)
        return compound.isomeric_smiles  # Returns the isomeric SMILES
    except:
        return None  # Returns None if CID lookup fails
    
# Iterate over selected OR-ligand pairs and generate JSON files
for _, row in or_merged.iterrows():
    cid = row["cid"]
    or_name = row["DL_OR"]
    ligand_name = row["odor_and_conc"]
    sequence = row["sequence"]
    
    name = f"{or_name}_ga_{cid}"

    json_data = {
        "name": name,
        "modelSeeds": [1],
        "sequences": [
            {
                "protein": {
                    "id": ["A"],
                    "sequence": sequence
                }
            },
            {
                "protein": {
                    "id": ["B"],
                    "sequence": "GGSLEVLFQGPSGNSKTEDQRNEEKAQREANKKIEKQLQKDKQVYRATHRLLLLGADNSGKSTIVKQMRILHGGSGGSGGTSGIFETKFQVDKVNFHMFDVGGQRDERRKWIQCFNDVTAIIFVVDSSDYNRLQEALNLFKSIWNNRWLRTISVILFLNKQDLLAEKVLAGKSKIEDYFPEFARYTTPEDATPEPGEDPRVTRAKYFIRDEFLRISTASGDGRHYCYPHFTCAVDTENARRIFNDCRDIIQRMHLRQYELL",
                }
            },                      
            {
                "ligand": {
                    "id": ["J"],
                    "smiles": get_smiles(cid)
                }
            }    
        ],
        "dialect": "alphafold3",
        "version": 1
    }

    # Save JSON file
    filename = f"/mnt/data2/Justice/AF3_files/AF3_input/pS6_screen/{name}.json"
    with open(filename, "w") as f:
        json.dump(json_data, f, indent=2)

    print(f"Saved: {filename}")



#### Generating .json files with template using consORs 

In [3]:
import os 
import sys 
import numpy as np 
import pandas as pd 

OR_LEARNING_PATH = os.path.join(os.getcwd().split('OR_learning')[0], 'OR_learning/')
sys.path.insert(0, os.path.join(OR_LEARNING_PATH, 'utils/'))

import pdb_functions as pu
import plot_functions as pf 
import SequenceAlignment_functions as sa 

# sys.path.insert(0, '/mnt/data2/Justice/alphafold3/src/')
# sys.path.insert(0, "/mnt/data2/Justice/alphafold3/src/alphafold3/parsers/cpp")

# from alphafold3 import structure


In [24]:
import importlib 

importlib.reload(pu)
importlib.reload(sa)

<module 'SequenceAlignment_functions' from '/mnt/data2/Justice/OR_learning/utils/SequenceAlignment_functions.py'>

In [ ]:
import gemmi

def filter_mmcif_by_chain(input_cif, output_cif, chain_id="A"):
    # Read the CIF file
    cif_doc = gemmi.cif.read(input_cif)
    cif_block = cif_doc.sole_block()  # Get the first block

    # Filter the _atom_site table (atomic coordinates)
    atom_site = cif_block.find_loop('_atom_site.')
    if atom_site:
        chain_col = atom_site.get_tag_position('_atom_site.auth_asym_id')
        if chain_col != -1:  # Ensure the column exists
            filtered_rows = [row for row in atom_site if row[chain_col] == chain_id]
            atom_site.set_all_values(filtered_rows)  # Update table with filtered rows

    # Save the modified CIF file
    with open(output_cif, "w") as f:
        f.write(cif_doc.as_string())  # Use as_string() instead of write_string()

# Example usage:
input_cif = "/mnt/data2/Justice/AF3_files/AF3_input/pS6_screen/template/original_cif/8uxy_consOR1.cif"
output_cif = "/mnt/data2/Justice/AF3_files/AF3_input/pS6_screen/template/test.cif"

filter_mmcif_by_chain(input_cif, output_cif, chain_id="A")

In [41]:
"""
First create separate template files for consORs 
"""

pu.filter_cif_by_chain('/mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/template/original_cif/8uxv_consOR51.cif', 
                         '/mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/template/8uxv_consOR51.cif', 
                         chain_id = ['A'])

pu.filter_cif_by_chain('/mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/template/original_cif/8uxy_consOR1.cif', 
                         '/mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/template/8uxy_consOR1.cif', 
                         chain_id = ['A'])


pu.cif_to_pdb('/mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/template/8uxv_consOR51.cif', 
              '/mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/template/converted_pdb/8uxv_consOR51.pdb')

pu.cif_to_pdb('/mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/template/8uxy_consOR1.cif', 
              '/mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/template/converted_pdb/8uxy_consOR1.pdb')


# Quickly make sure that chain X extracted from .cif is indeed Ga via visualization
# a = pu.load_pdb_coordinates('/mnt/data2/Justice/AF3_files/template/test_8uyq_consOR4.pdb')
# x = pu.load_pdb_coordinates('/mnt/data2/Justice/AF3_files/template/test_8uyq_Ga.pdb')

# pf.plot_coordinates([a[1], x[1]])

Converted PDB saved to: /mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/template/converted_pdb/8uxv_consOR51.pdb
Converted PDB saved to: /mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/template/converted_pdb/8uxy_consOR1.pdb


In [31]:
or_merged = pd.read_csv('/mnt/data2/Justice/OR_learning/output/Canonical_bc/AF3/pS6_screen/or_screen.csv', 
                        index_col = 0)
# or_merged.head()

In [38]:
# Get pdb files of OR in or_merged 

pdb_dir_path = '/mnt/data2/Justice/AF_files/AF_tmaligned_pdb/'
pdb_files = os.listdir(pdb_dir_path)
pdb_files = np.unique([_pdb for _olfr in np.unique(or_merged.id) for _pdb in pdb_files if _olfr == _pdb.split('_')[0]])

In [94]:
"""
Generate structural sequence alignment in order to get the alignment indices for AF3 .json files 

ORs are simply separeated to class I and class II for templates. 
class I : consOR51 as template 
class II: consOR1 as template 
"""

import json
import re 
import pubchempy as pcp

def get_smiles(cid):
    try:
        compound = pcp.Compound.from_cid(cid)
        return compound.isomeric_smiles  # Returns the isomeric SMILES
    except:
        return None  # Returns None if CID lookup fails

def map_aligned_indices(template_seq, query_seq):
    """
    Maps residue indices between a template (reference) and query sequence based on alignment.
    
    Parameters:
    - template_seq (str): Aligned template sequence (with gaps '-')
    - query_seq (str): Aligned query sequence (with gaps '-')
    
    Returns:
    - list of tuples (template_index, query_index)
    """
    template_idx = 0  # AF3 template requires O-based indices
    query_idx = 0
    mapping = []

    for t_res, q_res in zip(template_seq, query_seq):
        if t_res != '-' and q_res != '-':  # Both residues exist (aligned position)
            mapping.append((template_idx, query_idx))
        
        if t_res != '-':  # Increment template index if not a gap
            template_idx += 1
        if q_res != '-':  # Increment query index if not a gap
            query_idx += 1

    return mapping


# =============================
# >>> USING Ga MSA TEMPLATE <<<
# =============================
# Using previously calculatd Ga template and MSA 
import json
with open('/mnt/data2/Justice/AF3_files/AF3_out/consor51_ga_c9/consor51_ga_c9_data.json') as f:
    consor51_ga_c9_data = json.load(f)



Olfr_DL = np.load('/mnt/data2/Justice/OR_learning/files/Olfr_DL.npy', allow_pickle=True).item()

class_I  = ['51', '52', '55', '56']  # Aquatic-like ORs
class_II = ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14'] 

template_indices = {}
query_indices = {}
for _pdb in pdb_files: 
    _pdb_path = os.path.join(pdb_dir_path, _pdb)
    _OR = Olfr_DL.get(_pdb.split('_')[0], None)
    _OR_class = re.match(r'(Or\d+)', str(_OR)).group(0).replace('Or', '') 
    
    if _OR_class in class_I: 
        template_pdb = '/mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/template/converted_pdb/8uxv_consOR51.pdb'
        template_cif = 'template/8uxv_consOR51.cif' # Relative path via .json file 
    elif _OR_class in class_II: 
        template_pdb = '/mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/template/converted_pdb/8uxy_consOR1.pdb'
        template_cif = 'template/8uxy_consOR1.cif' # Relative path via .json file 
    # Align the AF2 structure to cryo 
    aligned_pdb = list(pu.tmalign_pdb(template_pdb, 
                             _pdb_path, 
                             save_pdb = False, return_coords = True))
    # Attach sequence into the list
    aligned_pdb.append(pu.load_pdb_coordinates(_pdb_path)[2])

    # Generate structural sequence alignment for AF3 indice 
    aligne_seq = sa.generate_sequence_alignment_pairs(
                                        reference = template_pdb, 
                                        targets   = [aligned_pdb], 
                                        load_pdb_fn = pu.load_pdb_coordinates, 
                                        labels = [_OR]
                                        )
                      
    # Generate alignment indices for query and template 
    template_indices, query_indices = zip(*map_aligned_indices(aligne_seq[_OR][0], 
                                                               aligne_seq[_OR][1]))
    # ==========================
    # >>> HARD CODED INDEXING <<<
    # ==========================
    # alphafold3 template_indice is 0-based, HOWEVER, it takes into account of the _atom_site table
    # Meaning even if the first ATOM in _atom_site table is SER, it's indexed on *_atom_site.label_seq_id - 1*
    # ex. if ATOM 5688 N N   . ALA E 5 21  ? 85.660  143.508 95.164  1.00 77.33 . . . --> template_indice = 21 - 1 = 20
    template_indices = tuple(_indice + 20 for _indice in template_indices) # Using hard coded +20, since consOR1 and consOR51 both starts at 21
    
    for _, row in or_merged[or_merged['DL_OR'] == _OR].iterrows():
        cid = row["cid"]
        or_name = row["DL_OR"]
        ligand_name = row["odor_and_conc"]
        sequence = row["sequence"]
        
        name = f"{or_name}_ga_{cid}_temp{os.path.basename(template_cif).split('_')[1].replace('.cif','')}"

        json_data = {
            "name": name,
            "modelSeeds": [1],
            "sequences": [
                {
                    "protein": {
                        "id": "A",
                        "sequence": sequence, 
                        "pairedMsa": "",
                        "unpairedMsa": "",
                        "templates":[
                            {
                                "mmcifPath": template_cif,
                                "queryIndices": list(query_indices),
                                "templateIndices": list(template_indices),
                            }
                            ] , 
                    }
                },
                {
                    # Ga protein sequence
                    # unpaired and pairedMSA is extracted from previously calculated Ga MSA to save time. 
                    "protein": {
                        "id": "B",
                        "sequence": "GGSLEVLFQGPSGNSKTEDQRNEEKAQREANKKIEKQLQKDKQVYRATHRLLLLGADNSGKSTIVKQMRILHGGSGGSGGTSGIFETKFQVDKVNFHMFDVGGQRDERRKWIQCFNDVTAIIFVVDSSDYNRLQEALNLFKSIWNNRWLRTISVILFLNKQDLLAEKVLAGKSKIEDYFPEFARYTTPEDATPEPGEDPRVTRAKYFIRDEFLRISTASGDGRHYCYPHFTCAVDTENARRIFNDCRDIIQRMHLRQYELL",
                        "pairedMsa": consor51_ga_c9_data['sequences'][1]['protein']['pairedMsa'],
                        "unpairedMsa": consor51_ga_c9_data['sequences'][1]['protein']['unpairedMsa'],
                        "templates": consor51_ga_c9_data['sequences'][1]['protein']['templates']
                    }
                },                      
                {
                    "ligand": {
                        "id": "J",
                        "smiles": get_smiles(cid)
                    }
                }    
            ], 
            "dialect": "alphafold3",
            "version": 1
        }
    
        # Save JSON file
        filename = f"/mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/{name}.json"
        with open(filename, "w") as f:
            json.dump(json_data, f, indent=2)

        print(f"Saved: {filename}")
        


Saved: /mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/Or5Aq1_ga_165675_tempconsOR1.json
Saved: /mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/Or5Aq1_ga_16666_tempconsOR1.json
Saved: /mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/Or5Aq1_ga_1049_tempconsOR1.json
Saved: /mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/Or4K47_ga_263626_tempconsOR1.json
Saved: /mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/Or4K47_ga_61653_tempconsOR1.json
Saved: /mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/Or4K47_ga_68490_tempconsOR1.json
Saved: /mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/Or7A36_ga_263626_tempconsOR1.json
Saved: /mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/Or7A36_ga_1049_tempconsOR1.json
Saved: /mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/Or2N1D_ga_7410_tempconsOR1.json
Saved: /mnt/data2/Justice/AF3_files/AF3_input/pS6_screen_template/Or2T26_ga_263626_tempconsOR1.json
Saved: /m

In [69]:

import json

with open('/mnt/data2/Justice/AF3_files/AF3_out/consor51_ga_c9/consor51_ga_c9_data.json') as f:
    consor51_ga_c9_data = json.load(f)
with open('/mnt/data2/Justice/AF3_files/AF3_out/or51e1_ga_c9/or51e1_ga_c9_data.json') as f:
    or51e1_ga_c9 = json.load(f)

In [ ]:
"""
Turns out Ga of different cryoEM have slightly different sequences. 
Slip using template for Ga for now. . . 
"""
# print(pu.load_pdb_coordinates('/mnt/data2/Justice/AF3_files/template/converted_pdb/8uxv_Ga.pdb')[2])
# print(pu.load_pdb_coordinates('/mnt/data2/Justice/AF3_files/template/converted_pdb/8uxy_Ga.pdb')[2])
# print(pu.load_pdb_coordinates('/mnt/data2/Justice/AF3_files/template/converted_pdb/8uy0_Ga.pdb')[2])
# print(pu.load_pdb_coordinates('/mnt/data2/Justice/AF3_files/template/converted_pdb/8uyq_Ga.pdb')[2])